# CSoNet Simulation — Run Notebook

Run cells **1 → 6** in order. Cell 7 is optional (plot only).

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/spatial_gossip_pd")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
for sub in ["results", "checkpoints", "plots", "code"]:
    (DRIVE_DIR / sub).mkdir(exist_ok=True)

print(f"Drive folder: {DRIVE_DIR}")
print("Subfolders ready: results/, checkpoints/, plots/, code/")


## 2. Install Ollama

In [ ]:
import os, sys, json, time, shutil, subprocess, urllib.request
from pathlib import Path

# -- Step 1: Clean up stale binaries --
print("Cleaning up old Ollama files...")
for p in ["/usr/local/bin/ollama", "/usr/bin/ollama", "/content/ollama", "/tmp/ollama_pkg"]:
    if os.path.exists(p):
        try:
            if os.path.islink(p): os.unlink(p)
            elif os.path.isdir(p): shutil.rmtree(p)
            else: os.remove(p)
        except Exception:
            pass

# -- Step 2: Fetch latest release from GitHub API --
print("Fetching latest Ollama release from GitHub...")
api_url = "https://api.github.com/repos/ollama/ollama/releases/latest"
req = urllib.request.Request(api_url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req) as response:
    release_data = json.loads(response.read().decode())

download_url = None
for asset in release_data["assets"]:
    if asset["name"] == "ollama-linux-amd64.tar.zst":
        download_url = asset["browser_download_url"]
        break
if not download_url:
    for asset in release_data["assets"]:
        name = asset["name"]
        if "linux" in name and "amd64" in name and "-mlx" not in name and "-rocm" not in name:
            download_url = asset["browser_download_url"]
            break
if not download_url:
    sys.exit("Could not find a Linux amd64 Ollama package on GitHub Releases.")

print(f"Downloading: {download_url}")
local_pkg = "/tmp/ollama_pkg"
subprocess.run(["wget", "-q", "--show-progress", "--timeout=120", "-O", local_pkg, download_url], check=True)

# -- Step 3: Extract to /usr --
print("Extracting...")
subprocess.run("apt-get install -y zstd -qq", shell=True)
subprocess.run(f"tar -C /usr -xf {local_pkg}", shell=True, check=True)

# -- Step 4: Verify binary --
ollama_path = "/usr/bin/ollama"
if not os.path.exists(ollama_path):
    ollama_path = shutil.which("ollama") or "/usr/local/bin/ollama"
if not os.path.exists(ollama_path):
    sys.exit("ollama binary not found after extraction.")
subprocess.run(f"chmod +x {ollama_path}", shell=True, check=True)
with open(ollama_path, "rb") as f:
    if f.read(4) != b"\x7fELF":
        sys.exit("Extracted binary has invalid format.")
print(f"Ollama installed at: {ollama_path}")

# -- Step 5: Redirect model storage to Drive --
ollama_model_dir = Path("/content/drive/MyDrive/spatial_gossip_pd/ollama_models")
ollama_model_dir.mkdir(exist_ok=True, parents=True)
ollama_home = Path.home() / ".ollama"
if ollama_home.exists() or ollama_home.is_symlink():
    if ollama_home.is_symlink(): ollama_home.unlink()
    else: subprocess.run(["rm", "-rf", str(ollama_home)])
ollama_home.symlink_to(ollama_model_dir)
print(f"Model storage -> {ollama_model_dir}")

# -- Step 6: Start server and wait --
print("Starting Ollama server...")
import requests as _req
env = os.environ.copy()
env["OLLAMA_HOST"] = "127.0.0.1:11434"
env["CUDA_VISIBLE_DEVICES"] = "0"
subprocess.Popen([ollama_path, "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for i in range(30):
    time.sleep(1)
    try:
        if _req.get("http://127.0.0.1:11434", timeout=2).status_code == 200:
            print(f"Ollama server ready after {i+1}s ✅")
            break
    except Exception:
        pass
else:
    print("WARNING: Server did not respond after 30s — run: !curl http://127.0.0.1:11434")


## 3. Pull Models

In [ ]:
import requests, subprocess

def is_model_pulled(model_name: str) -> bool:
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        pulled = [m["name"] for m in r.json().get("models", [])]
        return any(p.startswith(model_name.split(":")[0]) for p in pulled)
    except Exception:
        return False

MODELS_TO_PULL = [
    "mistral:7b-instruct",
    "llama3.1:8b",
    "qwen3:8b",
]

for model in MODELS_TO_PULL:
    if is_model_pulled(model):
        print(f"  ✓  {model}  (already cached)")
    else:
        print(f"  ↓  Pulling {model} ...")
        subprocess.run(["ollama", "pull", model], check=True)
        print(f"  ✓  {model}  done")


## 4. Copy Code from Drive

In [ ]:
import shutil, os, importlib, sys
from pathlib import Path

CODE_SRC  = DRIVE_DIR / "code"
CODE_DEST = Path("/content/spatial_gossip_pd")
CODE_DEST.mkdir(exist_ok=True)

# Copy all .py files from Drive to Colab RAM
py_files = list(CODE_SRC.glob("*.py"))
if not py_files:
    print("WARNING: No .py files found in Drive/code/")
    print("Please upload: config.py, data_structures.py, gossip.py, llm_agent.py,")
    print("               graph.py, runner.py, main.py, plotting.py")
else:
    for src in py_files:
        shutil.copy(src, CODE_DEST / src.name)
        print(f"  Copied {src.name}")

# Symlink results/, checkpoints/, plots/ -> Drive (persists across sessions)
for folder in ["results", "checkpoints", "plots"]:
    link = CODE_DEST / folder
    target = DRIVE_DIR / folder
    if link.is_symlink(): link.unlink()
    elif link.exists(): shutil.rmtree(link)
    link.symlink_to(target)
    print(f"  {folder}/ -> Drive/{folder}/")

# Clear stale module cache so Python picks up the fresh files
for mod in list(sys.modules.keys()):
    if mod in ("llm_agent", "runner", "main", "gossip", "graph", "plotting", "config", "data_structures"):
        del sys.modules[mod]

os.chdir(CODE_DEST)
print(f"\nWorking directory: {os.getcwd()}")


## 5. Start Watchdog (auto-restart Ollama if it crashes)

In [ ]:
import threading, subprocess, time, requests, os, shutil

def ollama_watchdog(interval: int = 30):
    """Background thread: restart Ollama if it stops responding."""
    def find_ollama():
        for p in ["/usr/bin/ollama", "/usr/local/bin/ollama"]:
            if os.path.exists(p):
                return p
        return shutil.which("ollama")

    while True:
        time.sleep(interval)
        try:
            if requests.get("http://localhost:11434", timeout=5).status_code == 200:
                continue  # Server is healthy
        except Exception:
            pass

        # Server is down or hung — kill and restart
        print("[WATCHDOG] Ollama not responding — restarting...")
        subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
        time.sleep(3)

        ollama = find_ollama()
        env = os.environ.copy()
        env["OLLAMA_HOST"] = "127.0.0.1:11434"
        env["CUDA_VISIBLE_DEVICES"] = "0"
        subprocess.Popen([ollama, "serve"], env=env,
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        for i in range(30):
            time.sleep(1)
            try:
                if requests.get("http://localhost:11434", timeout=2).status_code == 200:
                    print(f"[WATCHDOG] Ollama restarted after {i+1}s ✅")
                    break
            except Exception:
                pass

watchdog = threading.Thread(target=ollama_watchdog, args=(30,), daemon=True)
watchdog.start()
print("✅ Ollama watchdog started (checking every 30s)")


## 6. Run Simulation

> Change `"1"` in `sys.argv` to the config number you want (1–9).

In [ ]:
import sys, importlib
from pathlib import Path

sys.path.insert(0, str(CODE_DEST))

# Set config number to run (1-9)
sys.argv = ["main.py", "1"]

import main as main_module
importlib.reload(main_module)
main_module.main()


## 7. Plot Results *(optional)*

In [ ]:
import sys, importlib, shutil

sys.argv = ["main.py", "plot"]

import main as main_module
importlib.reload(main_module)
main_module.main()

# Zip plots for easy download
zip_path = DRIVE_DIR / "plots_export"
shutil.make_archive(str(zip_path), "zip", str(DRIVE_DIR / "plots"))
print(f"Plots zipped: {zip_path}.zip")
